ST 554 HW#10\
Hui Fang\
4/19/2026

# Part 1 - Creating Streaming Data Using Rate

Setup a data stream using the `"rate"` format.
Prior to starting the stream, set up a sequence of actions using appropriate functions from `pyspark.sql.functions`
that uses the `rate` data to 
- find the square root of the rate ‘value’ 
- find mod 4 of the rate ‘value’

To output this, create a `writeStream` that writes to ‘memory’ (`format("memory")`). Give the query a name
(`queryName("...")`) and start it!\
Let it run for about 30 seconds and then stop the query. Then output the entire table stored in the queryname (`spark.sql("select * from you_table_name").show()`).

## 1. Create the streaming DataFrame

I first create a Spark Session and create a streaming DataFrame using the built-in 'rate' source. This generates rows with (timestemp, value) at 1 row per second.

In [15]:
# import module
from pyspark.sql import SparkSession
# create a spark session
spark = SparkSession.builder.getOrCreate()
# create a streaming DataFrame using the built-in 'rate' source
rate_df = (
    spark.readStream
        .format("rate")
        .option("rowsPerSecond", 1)
        .load()
)

## 2. Conduct transformations

Before starting the stream, I applied transformation from `pyspark.sql.functions` to compute the square root of the 'value' column and the value modulo 4, which is a math operation that gives us the remainder when a number is divided by 4.

In [16]:
# import module
from pyspark.sql.functions import sqrt, col
# apply the required transformations
transformed = (
    rate_df
        .withColumn("sqrt_value", sqrt(col("value")))
        .withColumn("mod4_value", col("value") % 4)
)

## 3. Write the stream to memory

To store the output, I created a `writeStream` that writes to memory useing `format("memory")`. I named the the query "rate_table" and started the stream.

In [17]:
query = (
    transformed.writeStream
               .format("memory")
               .queryName("rate_table")
               .outputMode("append")
               .start()
)

26/04/19 12:31:01 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e1f0ef93-d546-44a8-89a9-dc8eac261f0b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/19 12:31:01 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


## 4. Stop the stream and query the in-memory table

After letting the stream run for about 30 seconds, I stopped the query and then used `spark.sql("select * from rate_table").show()` to display all rows stroed in the in-memory table. 

In [18]:
# stop the steam
query.stop()

# query the in-memory table
spark.sql("SELECT * FROM rate_table").show()

+--------------------+-----+------------------+----------+
|           timestamp|value|        sqrt_value|mod4_value|
+--------------------+-----+------------------+----------+
|2026-04-19 12:31:...|    0|               0.0|         0|
|2026-04-19 12:31:...|    1|               1.0|         1|
|2026-04-19 12:31:...|    2|1.4142135623730951|         2|
|2026-04-19 12:31:...|    3|1.7320508075688772|         3|
|2026-04-19 12:31:...|    4|               2.0|         0|
|2026-04-19 12:31:...|    5|  2.23606797749979|         1|
|2026-04-19 12:31:...|    6| 2.449489742783178|         2|
|2026-04-19 12:31:...|    7|2.6457513110645907|         3|
|2026-04-19 12:31:...|    8|2.8284271247461903|         0|
|2026-04-19 12:31:...|    9|               3.0|         1|
|2026-04-19 12:31:...|   10|3.1622776601683795|         2|
|2026-04-19 12:31:...|   11|   3.3166247903554|         3|
|2026-04-19 12:31:...|   12|3.4641016151377544|         0|
|2026-04-19 12:31:...|   13| 3.605551275463989|         

26/04/19 12:31:32 WARN DAGScheduler: Failed to cancel job group 94dfe550-01a2-4344-9416-d5b158444089. Cannot find active jobs for it.
26/04/19 12:31:32 WARN DAGScheduler: Failed to cancel job group 94dfe550-01a2-4344-9416-d5b158444089. Cannot find active jobs for it.


# Part 2 - Using data from a CSV with a Pipeline

There are six `bikeDetails` sub datasets available on the assignment link. The one named `bikeDetails_for_fit.csv`
should be read in as a spark (SQL) data frame. With this spark SQL data frame do the following
- use an SQLTransformer with the following statement (this does some log transforms, renames a variable, and creates a dummy variable from categorical variable):

`SELECT log(selling_price) as label, year, log(km_driven) as log_km_driven,
CASE WHEN owner = ’1st owner’ THEN 1 ELSE 0 END AS one_owner
FROM __THIS__`

- use a `VectorAssembler` to create a `features` column. The `features` column should include the `year`,
`log_km_driven`, and `one_owner` variables.
- create a `Pipeline` with the two steps above (`SQLTransformer` then `VectorAssembler`)
- fit this pipeline to the SQL data frame and save this as an object.

Now we want to set up a read stream to look for csv files placed into a folder (the five `bikeDetails_add*.csv`
files). When a csv comes in, we want to transform it using the fitted pipeline’s `.transform()` method! A
few notes:
- You’ll need a schema to set up the `readStream`. You can use the SQL data frame’s schema from above!
(`.schema` attribute)
- Each file you’ll be adding to the folder has a header!

We’ll just write the output to the ‘console’ using the ‘append’ mode.\
Once that is all set up, make sure your folder you are looking for the .csv files is empty. Then start the
query. Place the other files into the folder one at a time. You should see the output appear below your
query. Once you’ve done all five files, stop the query!\
This process will set us up well for using a fitted model to obtain predictions on streaming data!